# Prepare the EU27 material–climate analysis

Rebuild the panel from the fixed Eurostat snapshot, run the analysis checks, and create an analysis package tied to the exact Git commit.

In [ ]:
from pathlib import Path
import hashlib, os, subprocess, sys

REPOSITORY_URL = "https://github.com/calinadriancomes/eu27-material-climate-sensitivity.git"
repo = Path("/content/eu27-material-climate-sensitivity")
subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(repo)], check=True)
os.chdir(repo)
git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
manifest_sha = hashlib.sha256(Path("checksums.sha256").read_bytes()).hexdigest()
print("Git commit:", git_commit)
print("Repository manifest SHA-256:", manifest_sha)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-lock.txt"], check=True)
subprocess.run([sys.executable, "code/checks.py", "verify"], check=True)

In [ ]:
subprocess.run([sys.executable, "code/01_prepare_data.py"], check=True)
subprocess.run([sys.executable, "tests/test_ties.py"], check=True)
subprocess.run([sys.executable, "code/02_analyze_progress.py"], check=True)
subprocess.run([sys.executable, "tests/test_reproduction.py"], check=True)
subprocess.run([sys.executable, "code/05_build_extended_publication_evidence.py"], check=True)
subprocess.run([sys.executable, "tests/test_extended_publication_evidence.py"], check=True)


In [ ]:
output = Path("/content/eu27-material-climate-analysis.zip")
subprocess.run([sys.executable, "code/checks.py", "package-analysis", "--output", str(output), "--git-commit", git_commit, "--repository-manifest-sha", manifest_sha], check=True)
from google.colab import files
files.download(str(output))